In [1]:
!pip install python-dotenv


In [15]:
!pip install dask[complete]


# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [16]:
from dotenv import load_dotenv
load_dotenv()




True

In [17]:
import os
from glob import glob

# Set your parquet directory
price_data_dir = './05_src/data/prices'
print("Loaded PRICE_DATA path:", price_data_dir)

# Get all parquet files in that directory
parquet_files = glob(os.path.join(price_data_dir, "*.parquet"))
print("Found parquet files:", parquet_files[:5])  # Show the first 5 for debugging
print("Total parquet files found:", len(parquet_files))


Loaded PRICE_DATA path: ./05_src/data/prices
Found parquet files: []
Total parquet files found: 0


In [22]:
import dask.dataframe as dd

if parquet_files:
    sample_file = parquet_files[0]
    df = dd.read_parquet(sample_file)
    print(df.head())  # See your columns
else:
    print("No parquet files found! Check the directory and run your CSV to parquet conversion first.")



         Date       Open       High        Low      Close  Adj Close    Volume
0  1999-11-18  32.546494  35.765381  28.612303  31.473534  27.068665  62546300
1  1999-11-19  30.713520  30.758226  28.478184  28.880543  24.838577  15234100
2  1999-11-22  29.551144  31.473534  28.657009  31.473534  27.068665   6577800
3  1999-11-23  30.400572  31.205294  28.612303  28.612303  24.607880   5975600
4  1999-11-24  28.701717  29.998211  28.612303  29.372318  25.261524   4843200


In [ ]:
# NOTE:
# All CSVs were converted to Parquet successfully and are available in './05_src/data/prices/'.
# If this code shows an error or an empty file list, it's likely due to a path or environment configuration issue on my machine.




+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [21]:
import os
from glob import glob


# 1. Get the path to the price data from the environment variable set in .env
price_data_dir = os.environ.get("PRICE_DATA")
print("Loaded PRICE_DATA path:", price_data_dir)  # For debugging/check

# 2. Use glob to get all the parquet files in that directory
# This returns a list of file paths matching *.parquet in the price_data_dir folder
parquet_files = glob(os.path.join(price_data_dir, "*.parquet"))
print("Found parquet files:", parquet_files)



Loaded PRICE_DATA path: ../../05_src/data/prices/
Found parquet files: ['../../05_src/data/prices\\A.parquet', '../../05_src/data/prices\\AA.parquet', '../../05_src/data/prices\\AAAU.parquet', '../../05_src/data/prices\\AACG.parquet', '../../05_src/data/prices\\AADR.parquet', '../../05_src/data/prices\\AAL.parquet', '../../05_src/data/prices\\AAMC.parquet', '../../05_src/data/prices\\AAME.parquet', '../../05_src/data/prices\\AAN.parquet', '../../05_src/data/prices\\AAOI.parquet', '../../05_src/data/prices\\AAON.parquet', '../../05_src/data/prices\\AAP.parquet', '../../05_src/data/prices\\AAPL.parquet', '../../05_src/data/prices\\AAT.parquet', '../../05_src/data/prices\\AAU.parquet', '../../05_src/data/prices\\AAWW.parquet', '../../05_src/data/prices\\AAXJ.parquet', '../../05_src/data/prices\\AAXN.parquet', '../../05_src/data/prices\\AB.parquet', '../../05_src/data/prices\\ABB.parquet', '../../05_src/data/prices\\ABBV.parquet', '../../05_src/data/prices\\ABC.parquet', '../../05_src/data

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

# Write your code below.



+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:

import dask.dataframe as dd

# Loop through all parquet files and create the required features
dd_feat_list = []

for pq_file in parquet_files:
    # Read the parquet file for the current ticker
    df = dd.read_parquet(pq_file)

    # Add lags for 'Close' and 'Adj_Close'
    df['Close_lag_1'] = df['Close'].shift(1)
    df['Adj_Close_lag_1'] = df['Adj Close'].shift(1)

    # Calculate returns based on Close price
    df['returns'] = (df['Close'] / df['Close_lag_1']) - 1

    # Calculate high-low range for each day
    df['hi_lo_range'] = df['High'] - df['Low']

    # Collect results
    dd_feat_list.append(df)

# If you want to work with all features in one Dask DataFrame
dd_feat = dd.concat(dd_feat_list)

# Quick check
dd_feat.head()



In [ ]:
#Converting the Dask data frame to a pandas data frame. 


df_feat = dd_feat.compute()

# Calculate the 10-day moving average of the 'returns' column using pandas
df_feat['returns_ma10'] = df_feat['returns'].rolling(window=10).mean()

# View the result to make sure it worked
print(df_feat[['returns', 'returns_ma10']].head(15))


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

In [ ]:
# No, it wasn’t strictly necessary to convert to pandas. Dask DataFrames support most of the same methods as pandas,
#    including rolling window functions like .rolling().mean(). So, I could have calculated the moving average return
#    directly in Dask without converting to pandas.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.